# Local Inference & Comparison
This notebook downloads the fine-tuned adapters from GCS and runs inference locally using the `transformers` library.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from google.cloud import storage
import pandas as pd
import json
import os
import shutil
import gc
from sklearn.metrics import accuracy_score, classification_report

# --- CONFIGURATION ---
BUCKET_NAME = "mlops-22f3002292"

# REPLACE these with the actual output paths from your Vertex AI Tuning Job
# Example: "finetuning/output/v1_model/experiment_run/.../model"
GCS_MODEL_PATH_V1 = "finetuning/output/v1_model/REPLACE_WITH_ACTUAL_PATH/model"
GCS_MODEL_PATH_V2 = "finetuning/output/v2_model/REPLACE_WITH_ACTUAL_PATH/model"

LOCAL_DIR_V1 = "./models/v1"
LOCAL_DIR_V2 = "./models/v2"

print("CUDA Available:", torch.cuda.is_available())

In [ ]:
# Helper: Download Model from GCS
def download_model_from_gcs(gcs_path, local_dir):
    if os.path.exists(local_dir):
        print(f"Directory {local_dir} already exists. Skipping download.")
        return
    
    print(f"Downloading from gs://{BUCKET_NAME}/{gcs_path} to {local_dir}...")
    # Using gsutil for efficiency in notebook environment
    os.makedirs(local_dir, exist_ok=True)
    !gsutil -m cp -r gs://{BUCKET_NAME}/{gcs_path}/* {local_dir}
    print("Download complete.")

# Helper: Load Test Data
def load_test_data(gcs_path):
    client = storage.Client()
    blob = client.bucket(BUCKET_NAME).blob(gcs_path)
    content = blob.download_as_text()
    data = []
    for line in content.strip().split('\n'):
        data.append(json.loads(line))
    return data

test_data_v1 = load_test_data("finetuning/v1/test.jsonl")
test_data_v2 = load_test_data("finetuning/v2/test.jsonl")

In [ ]:
# Helper: Construct Prompt (Manually to match Reference)
def make_prompt(messages):
    # We extract user content. System prompt is usually embedded or prepended.
    user_content = next(m['content'] for m in messages if m['role'] == 'user')
    
    prompt = (
        "<start_of_turn>system\n"
        "Classify the flower based on its measurements into one of the following species: [Setosa, Versicolor, Virginica]\n"
        "<end_of_turn>\n"
        "<start_of_turn>user\n"
        f"{user_content}\n"
        "<end_of_turn>\n"
        "<start_of_turn>assistant\n"
    )
    return prompt

print(make_prompt(test_data_v1[0]['messages']))

In [ ]:
# Evaluation Function with Memory Management
def evaluate_local_model(model_local_path, test_data, model_name):
    print(f"\n=== Loading {model_name} from {model_local_path} ===")
    
    # Load Tokenizer & Model
    tokenizer = AutoTokenizer.from_pretrained(model_local_path)
    model = AutoModelForCausalLM.from_pretrained(
        model_local_path, 
        device_map="auto", 
        torch_dtype=torch.float16
    )
    
    preds = []
    trues = []
    
    print("Starting Inference...")
    for i, entry in enumerate(test_data):
        prompt = make_prompt(entry['messages'])
        target = next(m['content'] for m in entry['messages'] if m['role'] == 'assistant')
        
        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
        
        # Generate
        with torch.no_grad():
            outputs = model.generate(
                **inputs, 
                max_new_tokens=10, 
                do_sample=False, # Deterministic for evaluation
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id
            )
            
        # Decode only the new tokens
        output_text = tokenizer.decode(outputs[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True)
        cleaned_pred = output_text.strip().lower()
        
        preds.append(cleaned_pred)
        trues.append(target.lower())
        
        if i % 10 == 0: print(".", end="")

    print("\nDone.")
    
    # CLEANUP to free GPU memory for the next model
    del model
    del tokenizer
    torch.cuda.empty_cache()
    gc.collect()
    print("🧹 Memory cleared.")
    
    return trues, preds

In [ ]:
# --- 1. Evaluate V1 (Raw) ---
download_model_from_gcs(GCS_MODEL_PATH_V1, LOCAL_DIR_V1)
y_true_v1, y_pred_v1 = evaluate_local_model(LOCAL_DIR_V1, test_data_v1, "V1 (Raw)")

# --- 2. Evaluate V2 (Descriptive) ---
download_model_from_gcs(GCS_MODEL_PATH_V2, LOCAL_DIR_V2)
y_true_v2, y_pred_v2 = evaluate_local_model(LOCAL_DIR_V2, test_data_v2, "V2 (Descriptive)")

In [ ]:
def show_results(title, trues, preds):
    print(f"\n=== {title} ===")
    acc = accuracy_score(trues, preds)
    print(f"Accuracy: {acc:.2%}")
    # We force valid labels to avoid errors if model hallucinates
    print(classification_report(trues, preds, labels=['setosa', 'versicolor', 'virginica'], zero_division=0))

show_results("V1 Results", y_true_v1, y_pred_v1)
show_results("V2 Results", y_true_v2, y_pred_v2)

# Comparison Logic
acc1 = accuracy_score(y_true_v1, y_pred_v1)
acc2 = accuracy_score(y_true_v2, y_pred_v2)

print("\n--- Final Verdict ---")
if acc2 > acc1:
    print(f"Descriptive data (V2) performed better by {(acc2-acc1)*100:.2f}%")
elif acc1 > acc2:
    print(f"Raw numeric data (V1) performed better by {(acc1-acc2)*100:.2f}%")
else:
    print("Both models performed equally.")